# 03 - Mainnet CLMM

Concentrated-liquidity AMM analysis. This notebook is intentionally separate from CPMM because CLMM needs tick/active-liquidity state and cannot reuse the CPMM closed-form baseline.

Current inputs:
- `results/historical_clmm_decoded.csv` — decoded historical CLMM swap observations
- `results/historical_clmm_swaps_status.csv` — inclusion/exclusion ledger
- `results/historical_clmm_pipeline_summary.csv` — collection/decode funnel
- `results/historical_clmm_live_swaps.csv` — live-discovered CLMM swaps for short archive windows
- `results/historical_clmm_live_snapshots.csv` — repeated snapshots of live watchlisted CLMM accounts
- `results/historical_clmm_live_candidates.csv` — readiness filter over live decoded swaps

Attack-evaluation input:
- `results/historical_clmm_candidates.csv` — CLMM attack evaluation rows; rejected rows remain useful as methodology/status evidence


## Load data


In [1]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "helpers").exists():
        sys.path.insert(0, str(candidate))
        break
    if (candidate / "notebooks" / "helpers").exists():
        sys.path.insert(0, str(candidate / "notebooks"))
        break

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

from helpers import (
    best_conditions,
    blocker_table,
    data_readiness,
    find_repo_root,
    historical_counterfactual_summary,
    hypothesis_scorecard,
    load_inputs,
    plot_realized_heatmap,
    plot_sensitivity_lines,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

ROOT = find_repo_root()
inputs = load_inputs(ROOT)

clmm = inputs["frames"]["historical_clmm"]
clmm_decoded = inputs["frames"]["historical_clmm_decoded"]
clmm_status = inputs["frames"]["historical_clmm_swaps_status"]
clmm_summary = inputs["frames"]["historical_clmm_pipeline_summary"]
clmm_state_probe = inputs["frames"]["historical_clmm_state_probe"]
clmm_live_swaps = inputs["frames"]["historical_clmm_live_swaps"]
clmm_live_snapshots = inputs["frames"]["historical_clmm_live_snapshots"]
clmm_live_candidates = inputs["frames"]["historical_clmm_live_candidates"]
cpmm = inputs["frames"]["historical_cpmm"]

display(data_readiness(ROOT, inputs, [
    "historical_clmm_decoded",
    "historical_clmm_swaps_status",
    "historical_clmm_pipeline_summary",
    "historical_clmm_state_probe",
    "historical_clmm_live_swaps",
    "historical_clmm_live_snapshots",
    "historical_clmm_live_candidates",
    "historical_clmm",
    "historical_cpmm",
]))


,dataset,file,rows,columns,ready_for_final_numbers,note
0,historical_clmm_decoded,results/historical_clmm_decoded.csv,4,29,False,decode coverage only; not profitability evidence
1,historical_clmm_swaps_status,results/historical_clmm_swaps_status.csv,50,16,True,ok
2,historical_clmm_pipeline_summary,results/historical_clmm_pipeline_summary.csv,2,7,True,ok
3,historical_clmm_state_probe,results/historical_clmm_state_probe.csv,4,22,False,state requirements only; not profitability evi...
4,historical_clmm_live_swaps,results/clmm_1h/historical_clmm_live_swaps.csv,1741,22,True,ok
5,historical_clmm_live_snapshots,results/clmm_1h/historical_clmm_live_snapshots...,642,11,True,ok
6,historical_clmm_live_candidates,results/clmm_1h/historical_clmm_live_candidate...,214,29,True,ok
7,historical_clmm,results/clmm_1h/historical_clmm_candidates.csv,214,34,True,ok
8,historical_cpmm,results/historical_cpmm_candidates.csv,72,42,False,legacy schema: regenerate CSV


## Scope check


In [2]:
if not clmm.empty:
    if "model_status" in clmm:
        display(clmm["model_status"].value_counts().rename_axis("model_status").reset_index(name="rows"))
    if "rejection_reason" in clmm:
        display(clmm["rejection_reason"].fillna("evaluated").value_counts().rename_axis("rejection_reason").reset_index(name="rows"))
    display(historical_counterfactual_summary(clmm))
elif not clmm_decoded.empty:
    display(Markdown(
        f"Decoded `{len(clmm_decoded)}` CLMM swap observation(s), but no counterfactual candidate CSV exists yet. "
        "This is expected until historical PoolState/tick-array pre-state and CLMM replay validation are implemented."
    ))
    display(clmm_summary)
    if not clmm_state_probe.empty:
        display(clmm_state_probe[[
            "pool_label",
            "slot",
            "instruction_index",
            "required_account_count",
            "current_probe_slot",
            "current_pool_state_ok",
            "current_amm_config_ok",
            "current_tick_arrays_ok",
            "historical_state_available",
            "candidate_ready",
        ]].head(20))
    display(
        clmm_decoded[[
            "pool_label",
            "slot",
            "signature",
            "instruction_index",
            "swap_variant",
            "direction",
            "amount_in",
            "actual_amount_out",
            "historical_state_status",
        ]].head(20)
    )
else:
    display(Markdown(
        "No decoded CLMM observations exist yet. Run `cargo run -p fork --bin historical_clmm -- --pool clmm_wsol_usdc --limit-per-pool 50 run-all`."
    ))
    if not clmm_status.empty:
        display(clmm_status["analysis_status"].value_counts().rename_axis("analysis_status").reset_index(name="rows"))

if not clmm_live_swaps.empty:
    display(Markdown(
        f"Live collector observed `{len(clmm_live_swaps)}` signature row(s). "
        "Rows become useful for candidate construction only when all required accounts were known before discovery and a previous snapshot exists."
    ))
    display(clmm_live_swaps["status"].value_counts().rename_axis("status").reset_index(name="rows"))
    if not clmm_live_snapshots.empty:
        display(clmm_live_snapshots["account_role"].value_counts().rename_axis("account_role").reset_index(name="snapshots"))

if not clmm_live_candidates.empty:
    ready = clmm_live_candidates["live_candidate_ready"].astype(str).str.lower().isin(["true", "1"])
    display(Markdown(f"Live CLMM readiness rows: `{len(clmm_live_candidates)}`, ready: `{int(ready.sum())}`."))
    if "rejection_reason" in clmm_live_candidates:
        display(clmm_live_candidates.loc[~ready, "rejection_reason"].value_counts().rename_axis("rejection_reason").reset_index(name="rows"))


,model_status,rows
0,rejected,124
1,evaluated,90


,rejection_reason,rows
0,previous_snapshot_after_block_time,110
1,no_profitable_attack,90
2,missing_previous_snapshot,11
3,missing_tick_array_snapshot,1
4,victim_replay_mismatch,1
5,unsupported_base_output,1


,pool_type,pool_label,rows,profitable_rate,feasible_rate,realized_rate,median_net_profit
0,raydium_clmm,clmm_wsol_usdc,214,0.0,0.0,0.0,0.0


Live collector observed `1741` signature row(s). Rows become useful for candidate construction only when all required accounts were known before discovery and a previous snapshot exists.

,status,rows
0,tx_failed,1480
1,decoded,214
2,no_single_target_swap,37
3,decode_failed,10


,account_role,snapshots
0,remaining_tick_account,489
1,observation_state,51
2,amm_config,51
3,pool_state,51


Live CLMM readiness rows: `214`, ready: `92`.

,rejection_reason,rows
0,previous_snapshot_after_block_time,110
1,missing_previous_snapshot,11
2,missing_tick_array_snapshot,1


## Required CLMM candidate fields


In [3]:
required = pd.DataFrame([
    {"field": "pool_type", "why": "distinguish raydium_clmm/orca_whirlpool from cpmm"},
    {"field": "pool_label, pool_address", "why": "group results by selected pool"},
    {"field": "slot, signature, instruction_index", "why": "trace every counterfactual row back to a historical swap"},
    {"field": "amount_in, min_amount_out, actual_amount_out", "why": "victim size and slippage bound"},
    {"field": "sqrt_price_x64_before, liquidity_before, tick_current_before", "why": "CLMM state at victim pre-state"},
    {"field": "tick_arrays_before", "why": "needed for executable CLMM swap simulation across ticks"},
    {"field": "fee_rate, protocol_fee_rate", "why": "fee-aware profitability"},
    {"field": "tx_cost_per_leg", "why": "net profit, not just gross extraction"},
])
display(required)


,field,why
0,pool_type,distinguish raydium_clmm/orca_whirlpool from cpmm
1,"pool_label, pool_address",group results by selected pool
2,"slot, signature, instruction_index",trace every counterfactual row back to a histo...
3,"amount_in, min_amount_out, actual_amount_out",victim size and slippage bound
4,"sqrt_price_x64_before, liquidity_before, tick_...",CLMM state at victim pre-state
5,tick_arrays_before,needed for executable CLMM swap simulation acr...
6,"fee_rate, protocol_fee_rate",fee-aware profitability
7,tx_cost_per_leg,"net profit, not just gross extraction"


## CPMM vs CLMM comparison


In [4]:
if clmm.empty or cpmm.empty:
    display(Markdown("Counterfactual CPMM vs CLMM comparison waits for both historical CPMM and CLMM candidate outputs."))
    if not clmm_decoded.empty:
        display(Markdown("CLMM decode coverage is available, but it is not profitability evidence yet."))
else:
    cpmm_summary = historical_counterfactual_summary(cpmm).assign(family="CPMM")
    clmm_candidate_summary = historical_counterfactual_summary(clmm).assign(family="CLMM")
    display(pd.concat([cpmm_summary, clmm_candidate_summary], ignore_index=True))


,pool_type,pool_label,inclusion_reason,rows,profitable_rate,feasible_rate,realized_rate,median_net_profit,family
0,raydium_cpmm,wsol_debt,NaN,4,0.0,0.0,0.0,0.000000e+00,CPMM
1,raydium_cpmm,wsol_surge,decoded_pre_state_available,2,1.0,1.0,1.0,3.975213e+12,CPMM
2,raydium_cpmm,wsol_surge,NaN,60,0.0,0.0,0.0,0.000000e+00,CPMM
3,raydium_cpmm,wsol_useless,NaN,6,0.0,0.0,0.0,0.000000e+00,CPMM
4,raydium_clmm,clmm_wsol_usdc,NaN,214,0.0,0.0,0.0,0.000000e+00,CLMM


## Thesis-ready takeaways


In [5]:
if clmm.empty:
    lines = [
        "- CLMM profitability remains an empirical gap, not a result.",
        f"- Decoded CLMM observations available: `{len(clmm_decoded)}` rows.",
        f"- CLMM state probe rows available: `{len(clmm_state_probe)}` rows.",
        "- Next blocker: historical PoolState/tick-array pre-state and victim-swap replay validation.",
    ]
    display(Markdown("\n".join(lines)))
else:
    display(Markdown(f"- Historical CLMM counterfactual candidates loaded: `{len(clmm)}` rows."))


- Historical CLMM counterfactual candidates loaded: `214` rows.